In [3]:
from openai_client import OpenAIClient

open = OpenAIClient()

open.get_completion_from_messages('##### Как дела #####')


'User message, remember to respond in Italian: # Как дела #'

In [5]:
# Define a delimiter for separating reasoning steps
step_delimiter = "####"

# System prompt guiding the model through the reasoning process
system_prompt = f"""
Follow these steps to answer customer queries, using '{step_delimiter}' to delineate each step.

Step 1:{step_delimiter} Determine if the query pertains to a specific product rather than a general category.

Step 2:{step_delimiter} Identify if the product is among the listed items, including details such as brand, features, and price.

[Provide a list of products here]

Step 3:{step_delimiter} Assess any assumptions made by the customer regarding product comparisons or specifications.

Step 4:{step_delimiter} Verify the accuracy of these assumptions based on provided product information.

Step 5:{step_delimiter} Correct any misconceptions, referencing only the listed products, and respond in a courteous manner.
"""

# Example user queries
example_query_1 = "How does the BlueWave Chromebook compare to the TechPro Desktop in terms of cost?"
example_query_2 = "Are televisions available for sale?"

# Formulating query prompts for the model
query_prompts_1 = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': f"{step_delimiter}{example_query_1}{step_delimiter}"},
]

query_prompts_2 = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': f"{step_delimiter}{example_query_2}{step_delimiter}"},
]

In [9]:
# Retrieve the model's response for the first example query
response_to_query_1 = get_response_for_queries(query_prompts_1)
print(response_to_query_1)

# Retrieve the model's response for the second example query
response_to_query_2 = get_response_for_queries(query_prompts_2)
print(response_to_query_2)

system

Follow these steps to answer customer queries, using '####' to delineate each step.

Step 1:#### Determine if the query pertains to a specific product rather than a general category.

Step 2:#### Identify if the product is among the listed items, including details such as brand, features, and price.

[Provide a list of products here]

Step 3:#### Assess any assumptions made by the customer regarding product comparisons or specifications.

Step 4:#### Verify the accuracy of these assumptions based on provided product information.

Step 5:#### Correct any misconceptions, referencing only the listed products, and respond in a courteous manner.

user
####How does the BlueWave Chromebook compare to the TechPro Desktop in terms of cost?####


In [ ]:
# Extracting only the final response from the model's output
try:
    # The response from the model is assumed to be in a variable named 'response_to_query_2'
    # The 'split' method is used to divide the output into a list of segments based on the 'step_delimiter'
    # The '[-1]' selects the last item in this list, which is the final response after all reasoning steps
    final_response = response_to_query_1.split(step_delimiter)[-1].strip()
except Exception as error:
    # If any error occurs during the process (e.g., 'response_to_query_2' is not defined), 
    # a default error message is assigned to 'final_response'
    final_response = "Sorry, I'm having trouble right now, please try asking another question."

In [10]:
def evaluate_response_against_detailed_rubric(test_data, llm_response):
    """
    Evaluates the LLM's response against a detailed rubric, considering various aspects of the response
    including accuracy, relevance, and completeness based on the provided test data. This function
    aims to provide a nuanced evaluation by scoring the response on multiple criteria and offering
    actionable feedback.
    
    Args:
        test_data (dict): A dictionary containing the 'customer_query', 'context' (background information),
        and optionally 'expected_answers' to facilitate a more granular evaluation.
        llm_response (str): The response generated by the LLM to the customer query.
    
    Returns:
        dict: A dictionary containing the overall score, scores by criteria, and detailed feedback.
    """
    # Define the rubric criteria and initialize scores
    rubric_criteria = {
        'accuracy': {'weight': 3, 'score': None, 'feedback': ''},
        'relevance': {'weight': 2, 'score': None, 'feedback': ''},
        'completeness': {'weight': 3, 'score': None, 'feedback': ''},
        'coherence': {'weight': 2, 'score': None, 'feedback': ''}
    }
    total_weight = sum(criterion['weight'] for criterion in rubric_criteria.values())

    # Construct the evaluation prompt
    system_prompt = "Evaluate the customer service agent's response considering the provided context."
    evaluation_prompt = f"""\
    [Question]: {test_data['customer_query']}
    [Context]: {test_data['context']}
    [Expected Answers]: {test_data.get('expected_answers', 'N/A')}
    [LLM Response]: {llm_response}

    Evaluate the response based on accuracy, relevance to the query, completeness of the information provided,
    and the coherence of the text. Provide scores (0-10) for each criterion and any specific feedback.
    """

    # Assuming a function fetch_llm_evaluation to handle the evaluation process
    evaluation_results = fetch_llm_evaluation(system_prompt, evaluation_prompt)

    # Parse the evaluation results to fill in the rubric scores and feedback
    # This step assumes the evaluation results are structured in a way that can be programmatically parsed
    # For example, using a predetermined format or markers within the text
    parse_evaluation_results(evaluation_results, rubric_criteria)

    # Calculate the overall score based on the weighted average of the criteria scores
    overall_score = sum(criterion['score'] * criterion['weight'] for criterion in rubric_criteria.values()) / total_weight

    # Compile the detailed feedback and scores
    detailed_feedback = {criteria: {'score': rubric_criteria[criteria]['score'], 'feedback': rubric_criteria[criteria]['feedback']} for criteria in rubric_criteria}

    return {
        'overall_score': overall_score,
        'detailed_scores': detailed_feedback
    }

def fetch_llm_evaluation(system_prompt, evaluation_prompt):
    """
    Simulates fetching an LLM-based evaluation. This function would typically send a request to an LLM
    service with the evaluation prompts and return the LLM's response for processing.
    """
    # Placeholder for LLM call (we simulate it with dummy feedback)
    return "Accuracy: 8, Relevance: 7, Completeness: 9, Coherence: 8"

def parse_evaluation_results(evaluation_text, rubric_criteria):
    """
    Parses the evaluation text returned by the LLM and extracts scores and feedback for each criterion
    in the rubric. Updates the rubric_criteria dictionary in place.
    
    Args:
        evaluation_text (str): The text response from the LLM containing evaluation scores and feedback.
        rubric_criteria (dict): The dictionary of rubric criteria to be updated with scores and feedback.
    """
    # Example parsing logic, to be replaced with actual parsing of the LLM's response
    lines = evaluation_text.split(', ')
    for line in lines:
        criteria, score = line.split(': ')
        score = int(score)
        rubric_criteria[criteria.lower()]['score'] = score
        rubric_criteria[criteria.lower()]['feedback'] = f"Good score for {criteria.lower()}."

# Example test data and LLM response
test_data = {
    'customer_query': "How do I reset my password?",
    'context': "The customer is trying to reset their password on an online platform.",
    'expected_answers': "Steps to reset the password."
}

llm_response = "To reset your password, go to the settings page, click 'Forgot Password', and follow the instructions sent to your email."

# Evaluate the response
evaluation = evaluate_response_against_detailed_rubric(test_data, llm_response)

# Print out the evaluation results
print(f"Overall Score: {evaluation['overall_score']}")
for criterion, details in evaluation['detailed_scores'].items():
    print(f"{criterion.capitalize()}: Score - {details['score']}, Feedback - {details['feedback']}")


Overall Score: 8.1
Accuracy: Score - 8, Feedback - Good score for accuracy.
Relevance: Score - 7, Feedback - Good score for relevance.
Completeness: Score - 9, Feedback - Good score for completeness.
Coherence: Score - 8, Feedback - Good score for coherence.
